# C24 — La VAD prend-elle du bruit pour de la parole ? (D5)

webrtcvad (D5) répond « parole » sur des sons continus : pluie, chants d'oiseaux, sinusoïdes.
L'application prédit alors une émotion sur du bruit. Ce notebook mesure, sur CPU :

- **faux positifs** : ESC-50 sans les 6 classes vocales (crying_baby, sneezing, coughing,
  laughing, breathing, snoring), part des extraits avec au moins 0,5 s de « parole » ;
- **parole retrouvée** : 200 extraits de CREMA-D (graine 42), part des extraits dont au moins
  50 % de la durée est « parole ».

Options : webrtcvad agressivité 0 à 3 ; webrtcvad plus un contrôle d'énergie dans la bande
300–3 400 Hz ; Silero VAD (torch). Toutes passent par le même nettoyage des segments que
l'application (fusion, segments minimum, marge : `configs/preprocess.yaml`). Critères : au plus
10 % de faux positifs et au moins 95 % de parole retrouvée. Aucune option ne les tient tous
les deux : on retient celle qui retrouve le plus de parole parmi celles à au plus 10 % de faux
positifs. **Internet activé, dataset `ejlok1/cremad` attaché, sans GPU.**

## a. Démarrage

On clone le dépôt (branche `BRANCH`), on l'installe en mode éditable (I9), puis silero-vad
à la version figée de l'extra `[ml]` (torch est déjà installé sur Kaggle).

In [ ]:
import sys

BRANCH = "fix/c24-vad"
REPO_URL = "https://github.com/mentalist011101/TP_SIGNAL_DS.git"
CLONE_DIR = "/kaggle/working/TP_SIGNAL_DS"

!rm -rf {CLONE_DIR}
!git clone --depth 1 --branch {BRANCH} {REPO_URL} {CLONE_DIR}
# Kaggle tourne sous Python 3.13, au-delà de requires-python (<3.13) du dépôt.
%pip install -q --ignore-requires-python -e {CLONE_DIR}
%pip install -q "silero-vad==6.3"

sys.path.insert(0, CLONE_DIR + "/src")
print(sys.version)

## b. ffmpeg

`load_audio` décode avec ffmpeg (D1), comme dans l'application.

In [ ]:
import shutil
import time

if shutil.which("ffmpeg") is None:
    !apt-get -qq update && apt-get -qq install -y ffmpeg
print("ffmpeg :", shutil.which("ffmpeg"))
NOTEBOOK_START = time.time()

## c. ESC-50 et CREMA-D

ESC-50 est téléchargé depuis GitHub (comme C5) ; CREMA-D est le dossier `AudioWAV` du dataset
attaché qui contient exactement 7 442 WAV (comme C12). Liens dans `/kaggle/working/datasets/<id>`
(contrat des manifestes).

In [ ]:
import urllib.request
import zipfile
from pathlib import Path

ESC50_ARCHIVE_URL = "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip"
INPUT_DIR = Path("/kaggle/input")
DATASETS_DIR = Path("/kaggle/working/datasets")
CREMAD_EXPECTED_WAV = 7442


def download_esc50() -> Path:
    # Archive d'environ 600 Mo, supprimée dès qu'elle est décompressée.
    archive_path = DATASETS_DIR / "esc50.zip"
    urllib.request.urlretrieve(ESC50_ARCHIVE_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATASETS_DIR)
    archive_path.unlink()
    (DATASETS_DIR / "ESC-50-master").rename(DATASETS_DIR / "esc50")
    return DATASETS_DIR / "esc50"


def find_cremad_root() -> Path:
    for audio_folder in sorted(INPUT_DIR.glob("**/AudioWAV")):
        wav_count = len(list(audio_folder.glob("*.wav")))
        if wav_count == CREMAD_EXPECTED_WAV:
            return audio_folder.parent
    raise RuntimeError("CREMA-D introuvable : attacher le dataset ejlok1/cremad.")


DATASETS_DIR.mkdir(parents=True, exist_ok=True)
if not (DATASETS_DIR / "esc50").exists():
    download_esc50()
cremad_link = DATASETS_DIR / "cremad"
if not cremad_link.exists():
    cremad_link.symlink_to(find_cremad_root())
print("ESC-50 :", DATASETS_DIR / "esc50")
print("CREMA-D :", cremad_link.resolve())

## d. Extraits mesurés

Manifestes versionnés du dépôt. ESC-50 : 2 000 extraits moins les 6 classes vocales (240),
soit 1 760. CREMA-D : tirage fixe de 200 extraits, graine 42.

In [ ]:
from echolens.config import REPO_ROOT
from echolens.core.audio.preprocess_config import load_preprocess_config
from echolens.core.data.manifest import load_manifest
from echolens.training.vad_benchmark import (
    load_vad_benchmark_config,
    non_vocal_esc50_rows,
    sample_rows,
)

benchmark_config = load_vad_benchmark_config()
vad_config = load_preprocess_config().vad
manifests_dir = REPO_ROOT / "data" / "manifests"

esc50_rows = load_manifest(manifests_dir / "esc50.csv")
noise_rows = non_vocal_esc50_rows(esc50_rows, benchmark_config.excluded_esc50_classes)
cremad_rows = load_manifest(manifests_dir / "cremad.csv")
speech_rows = sample_rows(cremad_rows, benchmark_config.cremad_sample_size, benchmark_config.seed)

noise_classes = {row.label for row in noise_rows}
speech_actors = {row.speaker for row in speech_rows}
print("ESC-50 non vocal :", len(noise_rows), "extraits,", len(noise_classes), "classes")
print("CREMA-D tiré :", len(speech_rows), "extraits,", len(speech_actors), "acteurs")
print("Réglages de la VAD (D5) :", vad_config)

## e. Décisions brutes, calculées une fois

Pour chaque extrait : `load_audio` (mono 16 kHz, comme l'application, sans normalisation),
décisions de webrtcvad aux 4 agressivités (trames de 30 ms), part d'énergie dans la bande
300–3 400 Hz sur les mêmes trames (`core.features.band_power_share`, R5), probabilités de Silero
(tranches de 32 ms). Mises en cache dans `/kaggle/working` : une relance ne recalcule rien.

In [ ]:
import torch

from echolens.core.audio import load_audio
from echolens.core.audio.io import TARGET_SAMPLE_RATE
from echolens.core.data.manifest import audio_path
from echolens.core.models.silero_vad import (
    SILERO_CHUNK_SAMPLES,
    load_silero_model,
    silero_speech_probabilities,
)
from echolens.training.vad_benchmark import (
    clip_decisions,
    load_clip_decisions,
    save_clip_decisions,
)

OUTPUT_DIR = Path("/kaggle/working")
SILERO_FRAME_S = SILERO_CHUNK_SAMPLES / TARGET_SAMPLE_RATE
# Un seul fil de calcul : mesure reproductible, comme l'application sur un petit serveur.
torch.set_num_threads(1)
silero_model = load_silero_model()


def decide_rows(rows, cache_name):
    cache_path = OUTPUT_DIR / cache_name
    if cache_path.exists():
        return load_clip_decisions(cache_path)
    decisions = []
    for row in rows:
        signal = load_audio(audio_path(row, DATASETS_DIR))
        probabilities = silero_speech_probabilities(silero_model, signal)
        clip = clip_decisions(
            signal, row.label, vad_config, benchmark_config, probabilities, SILERO_FRAME_S
        )
        decisions.append(clip)
    save_clip_decisions(decisions, cache_path)
    return decisions


start = time.time()
noise_clips = decide_rows(noise_rows, "c24_esc50_decisions.json")
speech_clips = decide_rows(speech_rows, "c24_cremad_decisions.json")
elapsed = time.time() - start
print(f"{len(noise_clips)} + {len(speech_clips)} extraits en {elapsed:.0f} s")

## f. Tableau des options et choix

Faux positifs (ESC-50 non vocal) et parole retrouvée (CREMA-D) de chaque option. Aucune option
ne tient les deux critères fixés à l'avance ; on retient donc, parmi les options à au plus 10 %
de faux positifs, celle qui retrouve le plus de parole. Les seuils sont choisis sur les extraits
qui servent aussi à les mesurer : les chiffres de l'option retenue sont légèrement optimistes.

In [ ]:
from echolens.training.vad_benchmark import score_options, select_option

scores = score_options(noise_clips, speech_clips, vad_config, benchmark_config)
print(f"{'Option':32s} {'Faux positifs':>14s} {'Parole retrouvée':>17s}  Critères")
for score in scores:
    verdict = "oui" if score.passes else "non"
    fp_text = f"{score.false_positive_rate:14.1%}"
    recall_text = f"{score.speech_recall:17.1%}"
    print(f"{score.option.name:32s} {fp_text} {recall_text}  {verdict}")

selected = select_option(scores, benchmark_config.max_false_positive_rate)
if selected is None:
    print("Toutes les options dépassent la limite de faux positifs.")
else:
    print("Option retenue :", selected.option.name)

## g. Classes d'ESC-50 prises pour de la parole

Pour la VAD actuelle (webrtcvad 2), pour l'option de chaque famille qui a le moins de faux
positifs et pour l'option retenue : les classes qui en donnent le plus (40 extraits par classe).

In [ ]:
from echolens.training.vad_benchmark import false_positive_labels, most_frequent_labels

CURRENT_OPTION_NAME = "webrtcvad 2"
TOP_CLASS_COUNT = 10


def fewest_false_positives(method):
    family = [score for score in scores if score.option.method == method]
    best = family[0]
    for score in family[1:]:
        if score.false_positive_rate < best.false_positive_rate:
            best = score
    return best


highlighted = {}
for score in scores:
    if score.option.name == CURRENT_OPTION_NAME:
        highlighted["actuelle"] = score
for method in ["webrtcvad", "webrtcvad_band", "silero"]:
    highlighted["moins de faux positifs, " + method] = fewest_false_positives(method)
if selected is not None:
    highlighted["retenue"] = selected

false_positive_classes = {}
for role, score in highlighted.items():
    labels = false_positive_labels(noise_clips, score.option, vad_config, benchmark_config)
    top = most_frequent_labels(labels, TOP_CLASS_COUNT)
    false_positive_classes[score.option.name] = top
    print(f"{role} — {score.option.name} : {len(labels)} faux positifs")
    print("   ", top)

## g bis. Part de parole détectée sur CREMA-D

Le critère « au moins 50 % de la durée » compte aussi les silences de début et de fin des
extraits. Pour chaque option mise en avant, et pour celle qui retrouve le plus de parole avec au
plus 10 % de faux positifs : percentiles 5, 10, 25 et 50 de la part de chaque extrait détectée
comme parole.

In [ ]:
import numpy as np

from echolens.training.vad_benchmark import speech_shares

PERCENTILES = [5, 10, 25, 50]

within_false_positive_limit = []
for score in scores:
    if score.false_positive_rate <= benchmark_config.max_false_positive_rate:
        within_false_positive_limit.append(score)
if within_false_positive_limit:
    best_recall = within_false_positive_limit[0]
    for score in within_false_positive_limit[1:]:
        if score.speech_recall > best_recall.speech_recall:
            best_recall = score
    highlighted["meilleur rappel sous 10 % de faux positifs"] = best_recall
    print(
        "Meilleur rappel sous 10 % de faux positifs :",
        best_recall.option.name,
        f"({best_recall.false_positive_rate:.1%} ; {best_recall.speech_recall:.1%})",
    )

share_percentiles = {}
for score in highlighted.values():
    shares = speech_shares(speech_clips, score.option, vad_config, benchmark_config)
    values = np.percentile(shares, PERCENTILES)
    rounded = [round(float(value), 3) for value in values]
    share_percentiles[score.option.name] = dict(zip(PERCENTILES, rounded, strict=True))
    print(f"{score.option.name:32s}", rounded)

## h. Signaux de synthèse

Sinusoïdes pures et bruit blanc de 3 s : secondes de « parole » trouvées par chaque option. Ces
signaux servent aux tests unitaires de la segmentation.

In [ ]:
import numpy as np

from echolens.training.vad_benchmark import speech_seconds

SYNTHETIC_DURATION_S = 3.0
SINE_FREQUENCIES_HZ = [200, 440, 1000, 2000, 3000, 5000]
SINE_AMPLITUDES = [0.05, 0.3, 0.9]
NOISE_AMPLITUDES = [0.01, 0.05, 0.3]

sample_count = int(SYNTHETIC_DURATION_S * TARGET_SAMPLE_RATE)
times = np.arange(sample_count) / TARGET_SAMPLE_RATE
synthetic_signals = {}
for frequency in SINE_FREQUENCIES_HZ:
    for amplitude in SINE_AMPLITUDES:
        sine = amplitude * np.sin(2 * np.pi * frequency * times)
        synthetic_signals[f"sinus {frequency} Hz x {amplitude}"] = sine.astype(np.float32)
generator = np.random.default_rng(benchmark_config.seed)
for amplitude in NOISE_AMPLITUDES:
    noise = amplitude * generator.standard_normal(sample_count)
    noise = np.clip(noise, -1.0, 1.0)
    synthetic_signals[f"bruit blanc x {amplitude}"] = noise.astype(np.float32)

synthetic_seconds = {}
for name, signal in synthetic_signals.items():
    probabilities = silero_speech_probabilities(silero_model, signal)
    clip = clip_decisions(signal, name, vad_config, benchmark_config, probabilities, SILERO_FRAME_S)
    seconds_per_option = {}
    for score in scores:
        seconds = speech_seconds(clip, score.option, vad_config, benchmark_config)
        seconds_per_option[score.option.name] = round(seconds, 2)
    synthetic_seconds[name] = seconds_per_option

shown_options = [score.option.name for score in highlighted.values()]
print("Colonnes :", shown_options)
for name, seconds_per_option in synthetic_seconds.items():
    shown = [f"{seconds_per_option[option]:5.2f}" for option in shown_options]
    print(f"{name:26s}", "  ".join(shown))

## i. Résultats

`c24_vad_benchmark.json` (→ `training/results/`) : tableau des options, option retenue, classes
d'ESC-50 prises pour de la parole, signaux de synthèse. Les décisions brutes restent en sortie
du notebook.

In [ ]:
import json
import platform
from importlib.metadata import version

option_records = []
for score in scores:
    record = {
        "name": score.option.name,
        "method": score.option.method,
        "aggressiveness": score.option.aggressiveness,
        "band_threshold": score.option.band_threshold,
        "silero_threshold": score.option.silero_threshold,
        "silero_end_threshold": score.option.silero_end_threshold,
        "false_positive_rate": round(score.false_positive_rate, 4),
        "speech_recall": round(score.speech_recall, 4),
        "passes": score.passes,
    }
    option_records.append(record)

selected_name = None
if selected is not None:
    selected_name = selected.option.name

results = {
    "branch": BRANCH,
    "esc50_non_vocal_clips": len(noise_clips),
    "cremad_clips": len(speech_clips),
    "vad_config": vad_config.model_dump(),
    "benchmark_config": benchmark_config.model_dump(),
    "options": option_records,
    "selected": selected_name,
    "false_positive_classes": false_positive_classes,
    "cremad_speech_share_percentiles": share_percentiles,
    "synthetic_speech_seconds": synthetic_seconds,
    "torch": torch.__version__,
    "silero_vad": version("silero-vad"),
    "python": platform.python_version(),
    "runtime_s": round(time.time() - NOTEBOOK_START),
}
results_path = OUTPUT_DIR / "c24_vad_benchmark.json"
results_text = json.dumps(results, indent=2, ensure_ascii=False)
results_path.write_text(results_text, encoding="utf-8")
print("Écrit :", results_path, f"({results['runtime_s']} s au total)")